In [2]:
import pandas as pd
import numpy as np
from llama_index.llms.ollama import Ollama
from llama_index.llms.openai import OpenAI
from llama_index.core import PromptTemplate

from tqdm import tqdm

In [4]:
from dotenv import load_dotenv

load_dotenv("../../.env")

True

In [23]:
llm = OpenAI(model="gpt-4o-mini")

In [7]:
df_tv = pd.read_parquet("../../data/video_extractions/latvia_tv_with_scores.parquet")

In [229]:
(df_tv.score > 0.8).sum(), df_tv.is_environmental.sum(), (df_tv.score > 0.5).sum()

(241, 309, 309)

In [8]:
from typing import Literal, Optional, List, Union
from pydantic import BaseModel, Field
from llama_index.core.output_parsers import PydanticOutputParser
from llama_index.core.program import LLMTextCompletionProgram

In [30]:
classifier_prompt = """
You are an expert in economic and environmental policies, with a focus on identifying narratives related to these domains. 
I will provide you with an excerpt from a TV news transcript. Your task is to determine whether the transcription primarily 
deals with economic policy, environmental policy, or neither.

For each transcription, provide the following details in JSON format:

- "classification" - specify whether the transcription deals with "economic policy", "environmental policy", or "neither"
- "context" - a brief description of the context in which the policy is discussed (maximum 3 sentences)
- "justification" - an expert analysis explaining why the transcription was classified as such, based on the context

<transcription>
{transcription}
</transcription>
"""


class ArticleClassification(BaseModel):
    classification: Literal["economic policy", "environmental policy", "neither"] = (
        Field(..., description="The class of the transciption")
    )
    context: str = Field(
        ...,
        description="Short description of the context in which the policy is discussed (maximum 3 sentences)",
    )
    justification: str = Field(
        ...,
        description="expert analysis explaining why the transcription was classified as such",
    )

In [31]:
classifier = LLMTextCompletionProgram.from_defaults(
    output_parser=PydanticOutputParser(output_cls=ArticleClassification),
    prompt_template_str=classifier_prompt,
    verbose=True,
    llm=llm,
)

output = classifier(transcription=df_tv.translation.iloc[0])

results = {}
for idx, row in tqdm(df_tv.iterrows(), total=df_tv.shape[0]):
    output = classifier(transcription=row.translation)
    video_id = row.video_id
    results[video_id] = {
        "classification": output.classification,
        "context": output.context,
        "justification": output.justification,
    }

df_classification = pd.DataFrame.from_dict(data=results, orient="index")

 96%|█████████▋| 1760/1825 [1:13:17<02:42,  2.50s/it]


ValidationError: 1 validation error for ArticleClassification
classification
  Input should be 'economic policy', 'environmental policy' or 'neither' [type=literal_error, input_value='both', input_type=str]
    For further information visit https://errors.pydantic.dev/2.9/v/literal_error

In [32]:
df_classification = pd.DataFrame.from_dict(data=results, orient="index")

In [33]:
df_classification.loc[df_classification.classification != "neither"].shape

(701, 3)

In [34]:
economic_prompt = """
You are an expert in economic theories and environmental policies, with a focus on identifying conspiracy theories 
related to economic and environmental issues. I will provide you with an excerpt from a TV news transcript. 
Your task is to identify and extract claims related to conspiracy theories linked to the environment and economy, 
such as those involving the "Green Deal" or allegations of "selling out our economy."

For each claim, provide the following details in JSON format:

- "claim" - the specific statement or allegation to be analyzed
- "context" - a brief description of the context in which the claim was made (maximum 1 paragraph)
- "analysis" - an initial expert analysis of the potential disinformation of this claim based on the context
- "disinformation_score" - the disinformation score, see below
- "disinformation_category" - the category of disinformation, see below
- "quote" - the exact quote that corresponds to the allegation

For the scores "disinformation_score":
- "very low" = no problem, the allegation is not misleading or at risk. No need to investigate further
- "low" = allegation that would require verification and questioning, but on a subject that is not very important and significant in the context of ecological issues (example: lawnmowers)
- "medium" = problematic allegation on an important ecological subject (scientific, impacts, elections, politics, transport, agriculture, energy, food, democracy ...), which would really need to be verified, deconstructed, debunked, and questioned. Especially for fallacious opinions
- "high" = serious allegation, especially if it denies scientific consensus

For the categories of disinformation "disinformation_category":
- "consensus" = denial of scientific consensus
- "facts" = fact to verify, specify, or contextualize
- "narrative" = fallacious narrative or opinion that sows doubt (for example: "environmentalists want to take away our freedoms")
- "other"

<transcription>
{transcription}
</transcription>
"""


class ClaimAnalysis(BaseModel):
    claim: str = Field(..., description="The allegation to potentially verify")
    context: str = Field(
        ...,
        description="Reformulation of the context in which this allegation was made (maximum 1 paragraph)",
    )
    analysis: str = Field(
        ...,
        description="Initial analysis from the expert's point of view on the potential disinformation of this allegation based on the context",
    )
    disinformation_score: Literal["very low", "low", "medium", "high"] = Field(
        ..., description="The disinformation score"
    )
    disinformation_category: Literal["consensus", "facts", "narrative", "other"] = (
        Field(..., description="The category of disinformation")
    )
    quote: str = Field(
        ..., description="The exact quote that corresponds to the allegation"
    )


class TranscriptAnalysis(BaseModel):
    claims: list[ClaimAnalysis] = Field(
        ..., description="List of claim analyses from the transcript"
    )

In [36]:
df_to_analyze = df_tv.loc[
    df_tv.video_id.isin(
        df_classification.loc[df_classification.classification != "neither"].index
    )
]

program = LLMTextCompletionProgram.from_defaults(
    output_parser=PydanticOutputParser(output_cls=TranscriptAnalysis),
    prompt_template_str=economic_prompt,
    verbose=True,
    llm=llm,
)


results = []
for idx, row in tqdm(df_to_analyze.iterrows(), total=df_to_analyze.shape[0]):
    output = program(transcription=row.translation)
    video_id = row.video_id
    record = {"video_id": video_id}
    for idx, claim in enumerate(output.claims):
        record.update(
            {
                "claim_id": idx,
                "claim": claim.claim,
                "context": claim.context,
                "analysis": claim.analysis,
                "disinformation_score": claim.disinformation_score,
                "disinformation_category": claim.disinformation_category,
                "quote": claim.quote,
            }
        )
    results.append(record)

100%|██████████| 701/701 [47:58<00:00,  4.11s/it]  


In [42]:
df_analysis = pd.DataFrame.from_records(results).dropna()
df_analysis = df_analysis.merge(
    df_classification.loc[df_classification.classification != "neither", "classification"],
    left_on="video_id",
    right_index=True,
)

In [92]:
df_analysis.loc[(df_analysis.disinformation_score == "high") & (df_analysis.classification == "environmental policy")]

,video_id,claim_id,claim,context,analysis,disinformation_score,disinformation_category,quote,classification
620,VjsSwqC0oBQ,0.0,we absolutely do not notice the impact of clim...,The statement was made in the context of discu...,This claim downplays the scientific consensus ...,high,consensus,this time we absolutely do not notice the impa...,environmental policy


In [68]:
df_analysis.loc[df_analysis.video_id=="WfFZfGBVITs"].context.item()

"This claim was made in the context of discussing the potential consequences of climate change on national economies, particularly Latvia's."

In [86]:
print(".\n".join(df_to_analyze.loc[(df_to_analyze.video_id=="Dxq_zD1w8TY")].translation.item().split(".")))

The upcoming wind turbine park in Iekļeme, Lithuania has already become a familiar sight for the residents, including local farmers, who graze their herds peacefully close to the turbines.
 However, most of the turbines are still under construction.
 The first one is expected to start operating at the beginning of next year.
 Here, 44 turbines will generate 900 gigawatt-hours of electricity per year, with a maximum capacity of 300 megawatts.
 This is twice as much as all the wind parks in Latvia combined.
 Latvia remains at a range of 130 to 140 megawatts according to various sources.
 Unfortunately, this is indeed the situation.
 And the situation has not changed regrettably for several years.
 The newest wind park was commissioned about two years ago, but prior to that, I believe it had been more than ten years since any new projects were initiated in Latvia at all.
 When discussing the reasons for stagnation, officials highlight two blocks.
 Firstly, Latvia has comparatively stricte

In [91]:
translations = df_to_analyze.set_index("video_id")[["translation"]]
translations.translation = translations.translation.str.replace("\n", " ")
df_analysis.loc[df_analysis.disinformation_score == "medium"].merge(
    right=translations,
    left_on="video_id",
    right_index=True,
    how="left",
).to_csv("../../data/video_extractions/latvia_tv_claims_medium_score.csv")

df_analysis.loc[df_analysis.disinformation_score == "high"].merge(
    right=translations,
    left_on="video_id",
    right_index=True,
    how="left",
).to_csv("../../data/video_extractions/latvia_tv_claims_high_score.csv")